# Cross-Market Transfer Learning for Stock Return Prediction
Rutgers Business School · Machine Learning in Finance · Professor Serdar Dinc  
Team: Rithik Reddy Matta, Aditi Saumya, Dhanushi Patel, Avani Bhavsar

**Research notebook — no empirical findings have been produced yet.**
Test whether U.S. pretraining helps UK and Japanese forecasts when local histories are short.
The approved proposal supplies the research design; the notebook uses secure WRDS credential prompts and reuses completed downloads.

Read each numbered part in order. Set `RUN_MODE` below, run Parts 1–2 once to acquire data,
then run individual analysis cells as needed. “Restart kernel” clears Python variables but
**does not delete the disk cache**. No cell automatically refreshes a completed data snapshot.

Sources: [JKP WRDS guide](https://jkpfactors.com/jkp-wrds-guide),
[official characteristic metadata](https://github.com/bkelly-lab/jkp-data/blob/main/src/jkp/data/resources/factor_details.xlsx),
[JKP documentation](https://jkpfactors-data.s3.amazonaws.com/documents/Documentation.pdf).
The approximate counts and 174 columns in the proposal are reference values, not forced targets.

## PART 1 — Project Setup
### 1.1 Objective and hypotheses
H1: Fine-tuning U.S. weights improves local MSE and IC relative to local-only training.
H2: The benefit decreases with additional local history. H3: Attention and the ensemble
improve on local Elastic Net and XGBoost. Economic success additionally requires positive
net long-short returns under transparent cost scenarios. All claims require held-out evidence.

Install `requirements.txt` in a virtual environment and select that Jupyter kernel.
All data, model, and evaluation code is contained in this notebook. You do not need to open separate Python modules.

In [ ]:
import copy
import hashlib
import importlib.metadata
import itertools
import json
import os
import platform
import random
import re
import time
from datetime import datetime, timezone
from getpass import getpass
from io import BytesIO
from pathlib import Path

# Configure OpenMP before numerical imports to avoid mixed-runtime conflicts on macOS.
os.environ.setdefault(
    "OMP_NUM_THREADS",
    "1" if platform.system() == "Darwin" else os.getenv("SLURM_CPUS_PER_TASK", "8"),
)
os.environ.setdefault("MKL_NUM_THREADS", os.environ["OMP_NUM_THREADS"])

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
import seaborn as sns
import torch
from IPython.display import display
from pyarrow import Table
from scipy.stats import spearmanr
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import OneHotEncoder
from statsmodels.stats.multitest import multipletests
from torch import nn
from torch.nn import functional as F
from xgboost import XGBRegressor

In [ ]:
RUN_MODE = os.getenv("RUN_MODE", "smoke")  # Change to "full" for final research.
DATA_METHOD = os.getenv("DATA_METHOD", "wrds")  # "wrds" or "local"
PROJECT_DIR = Path.cwd()
RAW_DIR = Path(os.getenv("JKP_RAW_DIR", str(PROJECT_DIR / "data/raw")))
LOCAL_IMPORT_DIR = PROJECT_DIR / "data/local_import"  # USA/GBR/JPN.parquet
WRDS_TABLE = "contrib.global_factor"
DOWNLOAD_START = "1999-12-31"  # Predictor month for January 2000 target.
COUNTRIES = ["USA", "GBR", "JPN"]
TARGET_COUNTRIES = ["GBR", "JPN"]
HISTORIES = [1, 3, 5]
EXPERIMENTS = list(itertools.product(TARGET_COUNTRIES, HISTORIES))
NEURAL_APPROACHES = ["local", "zero_shot", "fine_tuned", "pooled"]
TARGET = "ret_exc_lead1m"
KEY = ["excntry", "id", "eom"]
SEEDS = [11, 29, 47] if RUN_MODE == "full" else [11]
MAX_EPOCHS = 100 if RUN_MODE == "full" else 2
PATIENCE = 10 if RUN_MODE == "full" else 2
BOOTSTRAPS = 49_999 if RUN_MODE == "full" else 49
COST_BPS = [0, 10, 25]
CPU_THREADS = (
    1
    if platform.system() == "Darwin"
    else min(8, int(os.getenv("SLURM_CPUS_PER_TASK", os.cpu_count() or 1)))
)
assert RUN_MODE in ["smoke", "full"] and DATA_METHOD in ["wrds", "local"]

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(CPU_THREADS)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = PROJECT_DIR / "results" / RUN_MODE
MODEL_ROOT = PROJECT_DIR / "models" / RUN_MODE
for directory in [OUTPUT_ROOT, MODEL_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)
print(f"Mode: {RUN_MODE} | device: {DEVICE} | CPU threads: {CPU_THREADS}")
print(
    "SMOKE OUTPUTS ARE DIAGNOSTICS, NOT RESEARCH RESULTS."
    if RUN_MODE == "smoke"
    else "Full specification selected."
)

## PART 2 — Data Acquisition
### 2.1 Official metadata and cache inspection
The official guide selects factor candidates through `abr_jkp`, not all numeric WRDS fields.
Public metadata are also cached. A manifest freezes the endpoint, column list and completed
country-year files. The cache is shared between smoke and full modes; smoke never triggers
a second data download. Transfer the **entire** `data/raw/` directory to Amarel.

### Data-cache functions
These small functions are defined once. They check local files before any login, save each
completed country-year download, and support loading a local WRDS export. **Run these cells
once after restarting the kernel; they do not connect to WRDS or download stock data.**

In [ ]:
GUIDE = "https://jkpfactors.com/jkp-wrds-guide"
METADATA_URL = (
    "https://raw.githubusercontent.com/bkelly-lab/jkp-data/main/"
    "src/jkp/data/resources/factor_details.xlsx"
)
SCREENS = ["common", "exch_main", "primary_sec", "obs_main"]
REQUIRED = ["excntry", "id", "eom", "ret_exc_lead1m", "size_grp", "me"] + SCREENS

In [ ]:
def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, default=str))
    temp.replace(path)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
def load_metadata(directory):
    """Download public metadata only if it is absent; stock data come from WRDS."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "factor_details.csv"
    if not path.exists():
        response = requests.get(METADATA_URL, timeout=120)
        response.raise_for_status()
        frame = pd.read_excel(BytesIO(response.content))
        temp = path.with_suffix(".tmp")
        frame.to_csv(temp, index=False)
        temp.replace(path)
    frame = pd.read_csv(path)
    if not {"abr_jkp", "name"}.issubset(frame):
        raise ValueError("Official factor metadata lack abr_jkp/name columns.")
    frame = frame[frame.abr_jkp.notna()].drop_duplicates("abr_jkp").copy()
    frame["abr_jkp"] = frame.abr_jkp.str.strip()
    if not frame.abr_jkp.str.fullmatch("[a-z][a-z0-9_]*").all():
        raise ValueError("Unexpected characteristic name in JKP metadata.")
    return frame

In [ ]:
def expected_parts(manifest):
    start, end = pd.Timestamp(manifest["start"]), pd.Timestamp(manifest["end"])
    for country in manifest["countries"]:
        for year in range(start.year, end.year + 1):
            yield country, year, f"{country}/{year}.parquet"

In [ ]:
def cache_status(raw_dir, verify_hash=False):
    """No network calls. A valid Parquet footer alone never marks a partial download complete."""
    raw_dir = Path(raw_dir)
    manifest_path = raw_dir / "manifest.json"
    if not manifest_path.exists():
        return None, ["snapshot manifest"]
    manifest = json.loads(manifest_path.read_text())
    missing = []
    for _, _, relative in expected_parts(manifest):
        path, entry = raw_dir / relative, manifest.get("parts", {}).get(relative)
        try:
            valid = bool(entry) and path.is_file()
            valid = valid and pq.read_metadata(path).num_rows == entry["rows"]
            valid = valid and path.stat().st_size == entry["bytes"]
            if valid and verify_hash:
                valid = sha256(path) == entry["sha256"]
        except (OSError, ValueError):
            valid = False
        if not valid:
            missing.append(relative)
    return manifest, missing

In [ ]:
def validate_snapshot(manifest, candidates, start, table):
    requested = {
        "table": table,
        "countries": list(COUNTRIES),
        "start": str(pd.Timestamp(start).date()),
        "candidates": list(candidates),
        "screens": {s: 1 for s in SCREENS},
    }
    for key, value in requested.items():
        if manifest[key] != value:
            raise ValueError(
                f"Frozen cache differs in {key}. Use a NEW RAW_DIR for a new snapshot."
            )

In [ ]:
def initialize_snapshot(connection, raw_dir, candidates, start, table="contrib.global_factor"):
    """Verify schema before selecting anything. Never guess unavailable WRDS columns."""
    if not re.fullmatch(r"[a-z_][a-z0-9_]*\.[a-z_][a-z0-9_]*", table):
        raise ValueError("Use a simple schema.table identifier.")
    library, name = table.split(".")
    description = connection.describe_table(library=library, table=name)
    available = set(description["name"])
    absent = sorted(set(REQUIRED) - available)
    if absent:
        raise ValueError(f"WRDS table is missing required fields: {absent}")
    selected = list(dict.fromkeys(REQUIRED + [x for x in candidates if x in available]))
    # ret is current-month total USD return and is used ONLY for turnover drift, never features.
    if "ret" in available:
        selected.append("ret")
    endpoint = connection.raw_sql(f"SELECT MAX(eom) AS endpoint FROM {table}")
    if pd.isna(endpoint.endpoint.iloc[0]):
        raise ValueError("The WRDS table has no dates.")
    last_complete = pd.Timestamp.today().normalize().replace(day=1) - pd.Timedelta(days=1)
    end = min(pd.Timestamp(endpoint.endpoint.iloc[0]), last_complete)
    manifest = {
        "version": 1,
        "source": GUIDE,
        "table": table,
        "countries": list(COUNTRIES),
        "start": str(pd.Timestamp(start).date()),
        "end": str(end.date()),
        "candidates": list(candidates),
        "columns": selected,
        "absent_candidates": sorted(set(candidates) - available),
        "screens": {s: 1 for s in SCREENS},
        "parts": {},
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    description.to_csv(Path(raw_dir) / "wrds_schema.csv", index=False)
    atomic_json(manifest, Path(raw_dir) / "manifest.json")
    return manifest

In [ ]:
def download_missing(connection, raw_dir, manifest, missing):
    """Stream each country-year, commit it atomically, then mark it complete."""
    raw_dir = Path(raw_dir)
    names = ", ".join('"' + name + '"' for name in manifest["columns"])
    conditions = " AND ".join(f"{s}=1" for s in SCREENS)
    for country, year, relative in expected_parts(manifest):
        if relative not in missing:
            continue
        lower = max(pd.Timestamp(manifest["start"]), pd.Timestamp(year=year, month=1, day=1))
        upper = min(pd.Timestamp(manifest["end"]), pd.Timestamp(year=year, month=12, day=31))
        sql = (
            f"SELECT {names} FROM {manifest['table']} WHERE {conditions} "
            "AND excntry=%(country)s AND eom BETWEEN %(lower)s AND %(upper)s"
        )
        params = {"country": country, "lower": str(lower.date()), "upper": str(upper.date())}
        target = raw_dir / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_suffix(".parquet.partial")
        writer, count = None, 0
        try:
            chunks = connection.raw_sql(
                sql, params=params, date_cols=["eom"], chunksize=50_000, return_iter=True
            )
            for chunk in chunks:
                chunk["eom"] = pd.to_datetime(chunk.eom)
                # Stable dtypes across chunks, including an all-missing chunk.
                for field in manifest["columns"]:
                    if field not in ["excntry", "id", "eom", "size_grp"]:
                        chunk[field] = pd.to_numeric(chunk[field], errors="raise").astype("float64")
                for field in ["excntry", "id", "size_grp"]:
                    chunk[field] = chunk[field].astype("string")
                arrow = Table.from_pandas(chunk, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(temporary, arrow.schema, compression="zstd")
                writer.write_table(arrow)
                count += len(chunk)
            if writer is None:
                raise ValueError(f"No schema returned for {relative}; not marking complete.")
            writer.close()
            writer = None
            temporary.replace(target)
            manifest["parts"][relative] = {
                "rows": count,
                "bytes": target.stat().st_size,
                "sha256": sha256(target),
            }
            atomic_json(manifest, raw_dir / "manifest.json")
            print(f"Cached {country} {year}: {count:,} rows")
        finally:
            if writer is not None:
                writer.close()
            temporary.unlink(missing_ok=True)
    return manifest

In [ ]:
def register_local_files(raw_dir, local_dir, candidates, start, table="contrib.global_factor"):
    """Import complete country Parquets exported from WRDS. No authentication required."""
    raw_dir, local_dir = Path(raw_dir), Path(local_dir)
    if (raw_dir / "manifest.json").exists():
        raise ValueError("Use a new RAW_DIR for local imports; an existing snapshot is protected.")
    frames = {}
    for country in COUNTRIES:
        path = local_dir / f"{country}.parquet"
        frame = pd.read_parquet(path)
        if not set(REQUIRED).issubset(frame):
            raise ValueError(f"{path} must contain {REQUIRED}")
        frame["eom"] = pd.to_datetime(frame.eom, errors="raise")
        if set(frame.excntry.dropna().unique()) != {country}:
            raise ValueError(f"Invalid country in {path}")
        frames[country] = frame
    end = min(frame.eom.max() for frame in frames.values())
    columns = sorted(set.intersection(*(set(f.columns) for f in frames.values())))
    manifest = {
        "version": 1,
        "source": "User-provided WRDS export",
        "table": table,
        "countries": list(COUNTRIES),
        "start": str(pd.Timestamp(start).date()),
        "end": str(end.date()),
        "candidates": list(candidates),
        "columns": columns,
        "screens": {s: 1 for s in SCREENS},
        "parts": {},
        "absent_candidates": sorted(set(candidates) - set(columns)),
    }
    for country, year, relative in expected_parts(manifest):
        f = frames[country]
        eligible = f[SCREENS].apply(pd.to_numeric, errors="raise").eq(1).all(axis=1)
        f = f.loc[
            eligible & f.eom.between(pd.Timestamp(start), end) & f.eom.dt.year.eq(year), columns
        ]
        path = raw_dir / relative
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_suffix(".partial")
        f.to_parquet(temp, index=False, compression="zstd")
        temp.replace(path)
        manifest["parts"][relative] = {
            "rows": len(f),
            "bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
    atomic_json(manifest, raw_dir / "manifest.json")
    return manifest

In [ ]:
def load_country(raw_dir, country, columns=None):
    """Read just one country into host RAM; no data are placed on GPU here."""
    manifest, missing = cache_status(raw_dir)
    if manifest is None or missing:
        raise FileNotFoundError(f"Incomplete WRDS cache: {missing[:5]}")
    files = [
        Path(raw_dir) / relative for c, _, relative in expected_parts(manifest) if c == country
    ]
    return pd.read_parquet(files, columns=columns)

In [ ]:
metadata = load_metadata(PROJECT_DIR / "data/metadata")
candidates = metadata.abr_jkp.tolist()
manifest, missing_parts = cache_status(RAW_DIR)
if manifest is not None:
    validate_snapshot(manifest, candidates, DOWNLOAD_START, WRDS_TABLE)
print(f"Official characteristic candidates: {len(candidates)}")
print(
    "Cache complete; WRDS login will be skipped."
    if not missing_parts
    else f"Data needed: {len(missing_parts)} cache entries (a new snapshot starts with one manifest)."
)
display(metadata[["abr_jkp", "name", "group"]].head())

### 2.2 Optional local WRDS export
For Method B, export the three countries from `contrib.global_factor` in WRDS, including
all four screening flags, identifiers, size fields, target and official characteristics.
Keep missing targets. Include predictor December 1999 onward and `ret` if available.
Save as `data/local_import/USA.parquet`, `GBR.parquet`, `JPN.parquet`.
The import is performed only when no manifest exists. All countries use their common endpoint.
Incomplete local snapshots fail visibly rather than attempting a WRDS login.

In [ ]:
if DATA_METHOD == "local" and manifest is None:
    manifest = register_local_files(
        RAW_DIR, LOCAL_IMPORT_DIR, candidates, DOWNLOAD_START, WRDS_TABLE
    )
    manifest, missing_parts = cache_status(RAW_DIR)
if DATA_METHOD == "local" and missing_parts:
    raise FileNotFoundError("Local cache is incomplete. Restore the missing country-year files.")

### 2.3 WRDS username and password — enter only in the prompts
**Run this cell to enter credentials; do not paste credentials into source code or chat.**
Both prompts are hidden, and neither value is printed or written to disk. Approve WRDS MFA
if requested. Your account must have the relevant data entitlements. This entire cell is
skipped when the cache is complete, including after restarting Jupyter.

No `.pgpass` file is created by this project. A failed connection stops without exposing
connection details. For batch jobs acquire the data interactively first.

In [ ]:
wrds_connection = None
if DATA_METHOD == "wrds" and missing_parts:
    import wrds
    from sqlalchemy import inspect as inspect_sql

    wrds_username = os.getenv("WRDS_USERNAME") or getpass("WRDS username (hidden): ")
    wrds_password = os.getenv("WRDS_PASSWORD") or getpass("WRDS password (hidden): ")
    try:
        wrds_connection = wrds.Connection(
            autoconnect=False,
            wrds_username=wrds_username,
            wrds_password=wrds_password,
            verbose=False,
        )
        # Fail directly instead of falling back to WRDS's optional credential-file prompts.
        wrds_connection._Connection__make_sa_engine_conn(raise_err=True)
        wrds_connection.insp = inspect_sql(wrds_connection.connection)
    except Exception:
        wrds_connection = None
        raise RuntimeError(
            "WRDS connection failed. Check credentials, MFA, access and network."
        ) from None
    finally:
        del wrds_username, wrds_password
else:
    print("No WRDS connection needed.")

### 2.4 Verify schema, download missing partitions, and close the connection
The guide documents all four screens as `=1`. They are applied in SQL and retained for
verification. Nothing is filtered on future labels. All necessary countries and years are
acquired now; even smoke mode caches the full extract. Country-year downloads use chunks,
atomic renaming and a manifest commit. An interrupted partition alone is downloaded again.
A new snapshot requires a new `RAW_DIR`; rerunning never refreshes the frozen endpoint.

In [ ]:
if wrds_connection is not None:
    try:
        if manifest is None:
            manifest = initialize_snapshot(
                wrds_connection, RAW_DIR, candidates, DOWNLOAD_START, WRDS_TABLE
            )
        manifest, missing_parts = cache_status(RAW_DIR)
        manifest = download_missing(wrds_connection, RAW_DIR, manifest, missing_parts)
    finally:
        wrds_connection.close()
        del wrds_connection
manifest, missing_parts = cache_status(RAW_DIR)
assert manifest is not None and not missing_parts, f"Incomplete data cache: {missing_parts}"
print(
    f"Frozen predictor endpoint: {manifest['end']}; downloaded stock data source: {manifest['source']}"
)
print("Unavailable official candidates:", manifest["absent_candidates"])

In [ ]:
raw_countries = {country: load_country(RAW_DIR, country) for country in COUNTRIES}
raw_summary = pd.DataFrame(
    [
        {
            "country": c,
            "screened_rows": len(f),
            "stocks": f.id.nunique(),
            "first_predictor_month": f.eom.min(),
            "last_predictor_month": f.eom.max(),
            "target_missing_pct": 100 * f[TARGET].isna().mean(),
            "columns": f.shape[1],
        }
        for c, f in raw_countries.items()
    ]
)
display(raw_summary)
print(
    "Proposal counts are descriptive reference values; dates/releases/screens can change actual counts."
)

## PART 3 — Data Cleaning
### 3.1 Validate the ex-ante stock universe
Duplicate keys and malformed dates fail instead of being silently discarded. JKP's lagged
characteristics are used as supplied; this does not eliminate historical database revision
or delisting coverage limitations. Targets remain missing. Infinite labels become missing;
infinite characteristics become missing and are reported. `ret` is reserved for portfolio
drift and never enters the feature list.

In [ ]:
def clean_country(frame, country):
    frame = frame.copy()
    frame["eom"] = pd.to_datetime(frame.eom, errors="raise")
    assert frame.eom.notna().all() and frame.eom.dt.is_month_end.all(), "Invalid month end"
    assert frame.id.notna().all() and frame.excntry.eq(country).all(), "Invalid stock/country"
    assert frame.eom.between(pd.Timestamp(DOWNLOAD_START), pd.Timestamp(manifest["end"])).all()
    frame["id"] = frame.id.astype("string")
    flags = frame[["common", "exch_main", "primary_sec", "obs_main"]].apply(
        pd.to_numeric, errors="raise"
    )
    frame = frame.loc[flags.eq(1).all(axis=1)].copy()
    assert not frame.duplicated(KEY).any(), "Duplicated stock-month keys"
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").replace([np.inf, -np.inf], np.nan)
    frame["target_month"] = frame.eom + pd.offsets.MonthEnd(1)
    assert (
        (frame.target_month.dt.to_period("M").astype(int) - frame.eom.dt.to_period("M").astype(int))
        .eq(1)
        .all()
    )
    numeric = [x for x in candidates if x in frame and pd.api.types.is_numeric_dtype(frame[x])]
    bad = int(np.isinf(frame[numeric].to_numpy(dtype=float)).sum())
    frame[numeric] = frame[numeric].replace([np.inf, -np.inf], np.nan)
    return frame.sort_values(["eom", "id"]).reset_index(drop=True), bad

In [ ]:
countries, quality_rows = {}, []
for country, frame in raw_countries.items():
    countries[country], infinite_features = clean_country(frame, country)
    quality_rows.append(
        {
            "country": country,
            "input_rows": len(frame),
            "eligible_rows": len(countries[country]),
            "infinite_features": infinite_features,
            "missing_labels": int(countries[country][TARGET].isna().sum()),
            "duplicate_keys": 0,
        }
    )
quality = pd.DataFrame(quality_rows)
display(quality)
del raw_countries

### 3.2 Training-only predictor selection shared across all comparisons
To keep inputs fixed across history lengths without letting a one-year experiment inspect
older local rows, select once using USA target years 2000–2017 and **only 2017** in both
local markets. A candidate must be at least 50% observed and vary within every one of these
three permitted samples. Labels are not consulted. This conservative common intersection
is a documented design choice; it may retain fewer than the proposal's approximate 130 inputs.
No size group, screen, country, ID, date or auxiliary return is added as an input.

In [ ]:
def selection_sample(frame, country):
    start = "2000-01-31" if country == "USA" else "2017-01-31"
    sample = frame.loc[frame.target_month.between(start, "2017-12-31")]
    if sample.empty:
        raise ValueError(f"No permitted predictor-selection observations for {country}")
    return sample


def choose_predictors(countries, metadata):
    summaries = {}
    for country, frame in countries.items():
        train = selection_sample(frame, country).reindex(columns=metadata.abr_jkp)
        summaries[country] = (train.isna().mean(), train.nunique(dropna=True))
    records = []
    for row in metadata.to_dict("records"):
        name = row["abr_jkp"]
        definition = row.get("name_new")
        record = {
            "predictor": name,
            "definition": row["name"] if pd.isna(definition) else definition,
            "theme": row.get("group"),
        }
        reasons = []
        for country, (missing, variation) in summaries.items():
            record[f"{country}_missing_pct"] = 100 * missing[name]
            if missing[name] > 0.5 or variation[name] <= 1:
                reason = "<50% available" if missing[name] > 0.5 else "no variation"
                reasons.append(f"{country}: {reason}")
        record.update(retained=not reasons, reason="; ".join(reasons))
        records.append(record)
    table = pd.DataFrame(records)
    selected = table.loc[table.retained, "predictor"].tolist()
    assert selected and not set(selected).intersection(KEY + [TARGET, "target_month", "ret"])
    return selected, table

In [ ]:
predictors, predictor_audit = choose_predictors(countries, metadata)
print(f"Retained {len(predictors)} of {len(candidates)} official candidates.")
display(predictor_audit.head(12))

### 3.3 Same-month ranking and categorical encoding
For each numerical feature use `(average_rank - 1)/(observed_count - 1) - 0.5` within a
country-month, with singleton/constant groups neutral at zero. Missing ranks receive zero.
No future labels select the ranking universe. This is a contemporaneous cross-sectional
transform, including at prediction time, rather than a transformation fitted on future data.
Any categorical candidate uses the mode and vocabulary from the same permitted training
samples as selection; unseen categories encode to all zeros. JKP's published characteristics
are normally numeric; the generic categorical path is retained and tested.

In [ ]:
def rank_numerical(frame, names):
    if not names:
        return pd.DataFrame(index=frame.index)
    groups = frame.groupby(["excntry", "eom"], observed=True)[names]
    ranks, counts = groups.rank(method="average"), groups.transform("count")
    ranked = (ranks - 1).div((counts - 1).replace(0, np.nan)) - 0.5
    return ranked.fillna(0).astype("float32")


def fit_categories(training, names):
    if not names:
        return {}, None
    modes = {name: training[name].dropna().astype(str).mode().iloc[0] for name in names}
    values = training[names].astype("string").fillna(modes).astype(str)
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    encoder.fit(values)
    return modes, encoder

In [ ]:
numeric_features = [
    x for x in predictors if all(pd.api.types.is_numeric_dtype(f[x]) for f in countries.values())
]
category_features = [x for x in predictors if x not in numeric_features]
selection_training = (
    pd.concat(
        [
            selection_sample(frame, country)[category_features]
            for country, frame in countries.items()
        ],
        ignore_index=True,
    )
    if category_features
    else pd.DataFrame()
)
category_modes, category_encoder = fit_categories(selection_training, category_features)
feature_names = numeric_features + (
    list(category_encoder.get_feature_names_out(category_features))
    if category_encoder is not None
    else []
)
for country, frame in countries.items():
    matrix = rank_numerical(frame, numeric_features)
    if category_features:
        values = frame[category_features].astype("string").fillna(category_modes).astype(str)
        encoded = category_encoder.transform(values)
        matrix = pd.concat(
            [
                matrix,
                pd.DataFrame(
                    encoded, index=frame.index, columns=feature_names[len(numeric_features) :]
                ),
            ],
            axis=1,
        )
    countries[country] = pd.concat([frame.drop(columns=predictors), matrix], axis=1)
    assert np.isfinite(countries[country][feature_names].to_numpy()).all()
del selection_training

### 3.4 Reproducible artifact namespace
Results are isolated by mode, frozen data hashes, code, dependencies and settings. Changing
these creates a new run directory rather than reusing stale models. Raw data are never
invalidated by model changes. Training and forecast artifacts inside an unchanged run are
loaded on rerun. Store persistent results outside scratch; scratch may hold Jupyter runtime files.

In [ ]:
settings = {
    "mode": RUN_MODE,
    "seeds": SEEDS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "bootstrap": BOOTSTRAPS,
    "cost_bps": COST_BPS,
    "predictors": predictors,
    "features": feature_names,
    "snapshot": manifest,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "device": str(DEVICE),
    "packages": {
        p: importlib.metadata.version(p)
        for p in ["scikit-learn", "xgboost", "pyarrow", "scipy", "wrds", "statsmodels"]
    },
}
notebook_source = json.loads((PROJECT_DIR / "Cross_Market_Transfer_Learning.ipynb").read_text())
source = "\n".join("".join(c.get("source", [])) for c in notebook_source["cells"])
settings["code_hash"] = hashlib.sha256(source.encode()).hexdigest()
RUN_ID = hashlib.sha256(json.dumps(settings, sort_keys=True, default=str).encode()).hexdigest()[:12]
RESULTS, MODELS = OUTPUT_ROOT / RUN_ID, MODEL_ROOT / RUN_ID
for name in ["predictions", "tables", "figures", "training"]:
    (RESULTS / name).mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
atomic_json(settings, RESULTS / "configuration.json")
atomic_json(predictors, RESULTS / "predictors.json")
joblib.dump((category_modes, category_encoder), RESULTS / "category_encoding.joblib")
predictor_audit.to_csv(RESULTS / "tables/table_2_predictor_audit.csv", index=False)
quality.to_csv(RESULTS / "tables/table_2_data_quality.csv", index=False)
raw_summary.to_csv(RESULTS / "tables/table_1_country_statistics.csv", index=False)
print("Run ID:", RUN_ID)

## PART 4 — Exploratory Data Analysis
### 4.1 Country statistics, availability, missingness and size
Eight purposeful figures including the timeline below. Outcome-dependent EDA uses training
returns only. Tables retain full country counts. Plot files are automatically saved.

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(RESULTS / "figures" / f"{name}.png", bbox_inches="tight")
    plt.show()
    plt.close()


eda = pd.concat(
    [f[["excntry", "id", "eom", "target_month", "size_grp", TARGET]] for f in countries.values()],
    ignore_index=True,
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.barplot(data=raw_summary, x="country", y="screened_rows", ax=axes[0])
sns.barplot(data=raw_summary, x="country", y="stocks", ax=axes[1])
axes[0].set_title("Eligible stock-month observations")
axes[1].set_title("Unique stocks")
save_figure("01_country_counts")
availability = eda.groupby(["target_month", "excntry"]).size().unstack()
availability.plot(
    figsize=(10, 3.5), ylabel="Eligible stocks", title="02 — Monthly stock availability"
)
save_figure("02_availability")
eda.assign(year=eda.target_month.dt.year).groupby(["year", "excntry"]).size().unstack().plot(
    figsize=(10, 3.5), ylabel="Stock-months", title="03 — Observations by target year"
)
save_figure("03_year_counts")

In [ ]:
missing_table = predictor_audit.set_index("predictor")[[f"{c}_missing_pct" for c in COUNTRIES]]
missing_table.mean(axis=1).nlargest(20).sort_values().plot.barh(
    figsize=(8, 5),
    xlabel="Mean training missingness (%)",
    title="04 — Least available candidate predictors",
)
save_figure("04_missingness")
size_shares = pd.crosstab(eda.excntry, eda.size_grp, normalize="index")
size_shares.plot.bar(
    stacked=True,
    figsize=(8, 3.5),
    ylabel="Share of eligible observations",
    title="05 — JKP size groups",
    rot=0,
)
save_figure("05_size_groups")
training_eda = eda.loc[eda.target_month.between("2000-01-31", "2017-12-31")].dropna(subset=[TARGET])
fig, ax = plt.subplots(figsize=(8, 3.5))
for c, frame in training_eda.groupby("excntry"):
    ax.hist(frame[TARGET].clip(-0.5, 0.5), bins=80, density=True, histtype="step", label=c)
ax.set(
    xlabel="Next-month USD excess return (tails clipped for display)",
    ylabel="Density",
    title="06 — Training return distributions",
)
ax.legend()
save_figure("06_training_returns")

In [ ]:
example_features = numeric_features[:10]
example = countries["USA"].loc[countries["USA"].target_month.le("2017-12-31"), example_features]
example = example.sample(min(50_000, len(example)), random_state=11)
sns.heatmap(example.corr(), cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("07 — Correlation of selected ranked training characteristics")
save_figure("07_characteristic_correlations")
lagged = training_eda[["excntry", "id", "target_month", TARGET]].copy()
lagged["target_month"] = lagged.target_month + pd.offsets.MonthEnd(1)
lagged = lagged.rename(columns={TARGET: "previous_return"})
serial = training_eda.merge(lagged, on=["excntry", "id", "target_month"], validate="one_to_one")
serial_correlations = serial.groupby("excntry")[[TARGET, "previous_return"]].corr().iloc[0::2, 1]
display(serial_correlations.rename("consecutive_month_return_correlation"))
print("Lagged returns here are descriptive training diagnostics; no future returns are features.")
del example, serial, lagged, training_eda, eda

## PART 5 — Train/Validation/Test Splits
### 5.1 Target-month alignment and local histories
Training labels end December 2017 (predictor November 2017); validation begins with
December 2017 predictors for January 2018 returns. December 2019 predictors belong to
the January 2020 test target. No validation observations are used for gradient updates.
Smoke mode retains every month but deterministically caps each country-month to 24 stocks
before model fitting; final research uses complete cross-sections. Smoke rankings were
computed over the full eligible universe first. This subset is a test of mechanics only.

In [ ]:
def partition(frame, period, years=None):
    if period == "train":
        first = "2000-01-31" if years is None else f"{2018 - years}-01-31"
        mask = frame.target_month.between(first, "2017-12-31")
    elif period == "validation":
        mask = frame.target_month.between("2018-01-31", "2019-12-31")
    elif period == "test":
        mask = frame.target_month.ge("2020-01-31")
    else:
        raise ValueError(period)
    result = frame.loc[mask].copy()
    assert not result.empty, f"Empty {period} partition"
    return result


def smoke_subset(frame, count=24):
    # Stable, label-independent stock selection. Never used in full mode.
    hashed = pd.util.hash_pandas_object(frame[["excntry", "id"]], index=False).to_numpy()
    return (
        frame.assign(_order=hashed)
        .sort_values("_order")
        .groupby(["excntry", "eom"], group_keys=False)
        .head(count)
        .drop(columns="_order")
        .sort_values(["eom", "id"])
        .reset_index(drop=True)
    )


def local_splits(country, years):
    frame = analysis_countries[country]
    return (
        partition(frame, "train", years),
        partition(frame, "validation"),
        partition(frame, "test"),
    )

In [ ]:
analysis_countries = {
    c: smoke_subset(f) if RUN_MODE == "smoke" else f for c, f in countries.items()
}
for c, frame in analysis_countries.items():
    splits = {p: partition(frame, p) for p in ["train", "validation", "test"]}
    month_sets = [set(x.target_month) for x in splits.values()]
    assert all(not a.intersection(b) for a, b in itertools.combinations(month_sets, 2))
    for years in HISTORIES if c != "USA" else []:
        local = partition(frame, "train", years)
        expected = pd.period_range(f"{2018 - years}-01", "2017-12", freq="M")
        assert set(local.target_month.dt.to_period("M")) == set(expected), (
            f"Incomplete {c} {years}-year history"
        )
    assert set(splits["validation"].target_month.dt.to_period("M")) == set(
        pd.period_range("2018-01", "2019-12", freq="M")
    )
    print(c, {p: (len(f), int(f[TARGET].notna().sum())) for p, f in splits.items()})
fig, ax = plt.subplots(figsize=(10, 2.8))
for i, (name, left, width, color) in enumerate(
    [
        ("US training", 2000, 18, "#35618f"),
        ("Local training (5 / 3 / 1 years)", 2013, 5, "#55a68b"),
        ("Validation", 2018, 2, "#d9a441"),
        ("Test", 2020, max(1, pd.Timestamp(manifest["end"]).year - 2019), "#b76765"),
    ]
):
    ax.barh(i, width, left=left, color=color)
    ax.text(left + 0.1, i, name, va="center", fontsize=9)
ax.set(
    xlabel="Target-return year",
    yticks=[],
    title="08 — Chronological split (local shorter histories end in 2017)",
)
save_figure("08_timeline")

In [ ]:
def month_weights(frame):
    counts = frame.groupby(["excntry", "eom"])[TARGET].transform("size")
    return (1 / counts).to_numpy()


def monthly_error(frame, prediction):
    error = pd.DataFrame(
        {
            "month": frame.target_month.to_numpy(),
            "error": (frame[TARGET].to_numpy() - np.asarray(prediction)) ** 2,
        }
    )
    return float(error.groupby("month").error.mean().mean())


def prediction_frame(frame, values):
    result = frame[KEY + ["target_month", TARGET]].copy()
    result["prediction"] = np.asarray(values)
    assert np.isfinite(result.prediction).all()
    assert not result.duplicated(KEY).any()
    return result


def prediction_tag(country, years, approach, seed=None):
    history = "" if approach == "zero_shot" else f"_{years}"
    suffix = "" if seed is None else f"_seed{seed}"
    return f"{country}{history}_{approach}{suffix}"


def save_predictions(model, frame, tag, split):
    path = RESULTS / "predictions" / f"{tag}_{split}.parquet"
    if path.exists():
        return
    values = (
        predict_network(model, frame)
        if isinstance(model, nn.Module)
        else model.predict(frame[feature_names])
    )
    temporary = path.with_suffix(".partial")
    prediction_frame(frame, values).to_parquet(temporary, index=False)
    temporary.replace(path)

## PART 6 — Baseline Models
### 6.1 Candidate settings and month-weighted validation
Each country/history is a tuning exercise: 12 Elastic Net settings, 12 XGBoost settings,
and at most 8 neural settings. Smoke mode uses one of each. Baseline training weights
sum equally over months. Validation selects mean monthly MSE. No test data enter selection.
XGBoost uses histogram trees and CUDA when available, with explicit CPU fallback otherwise.

In [ ]:
EN_GRID = [dict(alpha=a, l1_ratio=r) for a in [1e-5, 1e-4, 1e-3, 1e-2] for r in [0.1, 0.5, 0.9]]
XGB_GRID = [
    dict(max_depth=d, learning_rate=lr, n_estimators=n)
    for d in [2, 4, 6]
    for lr in [0.03, 0.1]
    for n in [100, 300]
]
NN_GRID = [
    dict(width=64, blocks=2, dropout=0.1, lr=1e-3, weight_decay=1e-4),
    dict(width=64, blocks=1, dropout=0.1, lr=3e-4, weight_decay=1e-4),
    dict(width=64, blocks=2, dropout=0.2, lr=3e-4, weight_decay=1e-3),
    dict(width=128, blocks=1, dropout=0.1, lr=3e-4, weight_decay=1e-4),
    dict(width=128, blocks=2, dropout=0.1, lr=1e-4, weight_decay=1e-3),
    dict(width=128, blocks=2, dropout=0.2, lr=3e-4, weight_decay=1e-4),
    dict(width=64, blocks=1, dropout=0.2, lr=1e-3, weight_decay=1e-3),
    dict(width=128, blocks=1, dropout=0.2, lr=1e-3, weight_decay=1e-3),
]
if RUN_MODE == "smoke":
    EN_GRID, XGB_GRID, NN_GRID = (
        EN_GRID[:1],
        [dict(max_depth=2, learning_rate=0.1, n_estimators=8)],
        NN_GRID[:1],
    )
assert max(map(len, [EN_GRID, XGB_GRID, NN_GRID])) <= 20
atomic_json(
    {"elastic_net": EN_GRID, "xgboost": XGB_GRID, "transformer": NN_GRID},
    RESULTS / "candidate_grids.json",
)

In [ ]:
def fit_baseline(family, train, validation, tag):
    path, report_path = MODELS / f"{tag}.joblib", RESULTS / "training" / f"{tag}.json"
    if path.exists() and report_path.exists():
        return joblib.load(path), json.loads(report_path.read_text())
    train, validation = train.dropna(subset=[TARGET]), validation.dropna(subset=[TARGET])
    if train.empty or validation.empty:
        raise ValueError("Baseline fitting requires observed training and validation returns.")
    grid = EN_GRID if family == "ElasticNet" else XGB_GRID
    best_model, best_score, trials = None, np.inf, []
    for params in grid:
        if family == "ElasticNet":
            model = ElasticNet(**params, max_iter=5000, tol=1e-6, selection="cyclic")
        else:
            model = XGBRegressor(
                **params,
                objective="reg:squarederror",
                tree_method="hist",
                device=str(DEVICE),
                n_jobs=CPU_THREADS,
                random_state=11,
            )
        model.fit(train[feature_names], train[TARGET], sample_weight=month_weights(train))
        score = monthly_error(validation, model.predict(validation[feature_names]))
        trials.append({**params, "validation_monthly_mse": score})
        if score < best_score:
            best_model, best_score, best_params = model, score, params
    report = {
        "family": family,
        "checkpoint": path.name,
        "parameters": best_params,
        "validation_monthly_mse": best_score,
        "candidates": trials,
        "trainable_weights": int(best_model.coef_.size + 1) if family == "ElasticNet" else None,
    }
    temporary = path.with_suffix(".tmp")
    joblib.dump(best_model, temporary)
    temporary.replace(path)
    atomic_json(report, report_path)
    return best_model, report

In [ ]:
baseline_reports = []
for country, years in EXPERIMENTS:
    train, validation, test = local_splits(country, years)
    for family in ["ElasticNet", "XGBoost"]:
        tag = prediction_tag(country, years, family)
        model, report = fit_baseline(family, train, validation, tag)
        for split, sample in [("validation", validation), ("test", test)]:
            save_predictions(model, sample, tag, split)
        baseline_reports.append(
            {"country": country, "history": years, "approach": family, "seed": None, **report}
        )
    print(f"Baselines complete: {country}, {years} year(s)")

## PART 7 — Transformer Architecture
### 7.1 Full country-month attention, without position encodings
Each stock is one token. There are no time or stock-order embeddings. Every attention
query sees all stocks of that same country-month. Separate months/countries never attend
to each other. PyTorch scaled-dot-product attention chooses a supported efficient backend;
no independent stock chunks, token truncation or sparse approximation occurs in full mode.
The architecture is permutation equivariant in evaluation mode (dropout disabled).
[PyTorch attention documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html).

In [ ]:
class CrossStockBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.heads, self.dropout = 4, dropout
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.qkv, self.projection = nn.Linear(width, 3 * width), nn.Linear(width, width)
        self.feedforward = nn.Sequential(
            nn.Linear(width, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, width)
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, tokens):
        count, width = tokens.shape
        qkv = self.qkv(self.norm1(tokens)).reshape(count, 3, self.heads, width // self.heads)
        q, k, v = qkv.permute(1, 2, 0, 3).unbind(0)
        attended = F.scaled_dot_product_attention(
            q.unsqueeze(0),
            k.unsqueeze(0),
            v.unsqueeze(0),
            dropout_p=self.dropout if self.training else 0.0,
        ).squeeze(0)
        attended = attended.transpose(0, 1).reshape(count, width)
        tokens = tokens + self.drop(self.projection(attended))
        return tokens + self.drop(self.feedforward(self.norm2(tokens)))


class CrossStockTransformer(nn.Module):
    def __init__(self, input_features, width=64, blocks=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_features, width)
        self.blocks = nn.ModuleList([CrossStockBlock(width, dropout) for _ in range(blocks)])
        self.output = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 1))

    def forward(self, features):
        tokens = self.embedding(features)
        for block in self.blocks:
            tokens = block(tokens)
        return self.output(tokens).squeeze(-1)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Exact resumption is supported; CUDA kernel differences can affect bitwise reproducibility.
    torch.backends.cudnn.benchmark = False


def make_network(parameters):
    return CrossStockTransformer(
        len(feature_names), **{k: parameters[k] for k in ["width", "blocks", "dropout"]}
    ).to(DEVICE)


def month_batches(frame):
    # Host arrays only; the current cross-section alone is copied to the accelerator.
    x = frame[feature_names].to_numpy(dtype=np.float32)
    y = frame[TARGET].to_numpy(dtype=np.float32)
    groups = [
        (str(c), str(m), idx)
        for (c, m), idx in frame.reset_index(drop=True)
        .groupby(["excntry", "eom"], sort=True)
        .indices.items()
    ]
    return x, y, groups

### 7.2 Memory benchmark and failure policy
Benchmark the largest available cross-section (features only; benchmark weights discarded) with forward/backward passes before tuning.
Report stock count, time and CUDA peak allocation. Attention remains quadratic in compute;
backend choice determines peak memory. If an allocation fails, stop and request more memory
or use a separately documented method; never silently change the experiment.
Smoke mode exercises a small real cross-section, not a realistic full memory benchmark.

In [ ]:
largest_frame = max(analysis_countries.values(), key=lambda f: f.groupby("eom").size().max())
largest_month = largest_frame.groupby("eom").size().idxmax()
benchmark_x = torch.tensor(
    largest_frame.loc[largest_frame.eom.eq(largest_month), feature_names].to_numpy(np.float32),
    device=DEVICE,
)
benchmark_rows = []
for width in sorted({p["width"] for p in NN_GRID}):
    network = make_network(dict(width=width, blocks=2, dropout=0.2))
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    started = time.perf_counter()
    try:
        network(benchmark_x).square().mean().backward()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    except torch.OutOfMemoryError:
        raise RuntimeError(
            "Full-stock attention exceeds this device. Increase allocated memory; no stocks were chunked."
        ) from None
    benchmark_rows.append(
        {
            "width": width,
            "stocks": len(benchmark_x),
            "seconds": time.perf_counter() - started,
            "cuda_peak_gb": torch.cuda.max_memory_allocated() / 1e9
            if DEVICE.type == "cuda"
            else None,
        }
    )
display(pd.DataFrame(benchmark_rows))
pd.DataFrame(benchmark_rows).to_csv(RESULTS / "tables/attention_benchmark.csv", index=False)
del network, benchmark_x, largest_frame
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
@torch.no_grad()
def predict_network(network, frame):
    network.eval()
    x, _, groups = month_batches(frame)
    values = np.empty(len(frame), dtype=np.float32)
    for _, _, index in groups:
        features = torch.as_tensor(x[index], device=DEVICE)
        values[index] = network(features).cpu().numpy()
    return values


def network_validation(network, frame):
    return monthly_error(frame, predict_network(network, frame))


def atomic_checkpoint(state, path):
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    temporary.replace(path)

### 7.3 Month and country balancing
A training epoch computes the mean loss for each eligible country-month, with missing
labels masked **after** attention. The gradient is averaged over months, and over countries
in pooled training, before a single AdamW update per epoch. This optimizes the documented
equal-month objective without letting larger cross-sections or the longer U.S. history
dominate pooling. The learning-rate grid is defined for this full-epoch update schedule.
Validation is evaluated only after updates; test observations are absent from this function.
Checkpoints save model, optimizer, best weights, epoch, losses and RNG states.

In [ ]:
def fit_network(train, validation, parameters, seed, tag, initial_state=None):
    seed_everything(seed)
    path = MODELS / f"{tag}.pt"
    network = make_network(parameters)
    if initial_state is not None:
        network.load_state_dict(initial_state)
    optimizer = torch.optim.AdamW(
        network.parameters(), lr=parameters["lr"], weight_decay=parameters["weight_decay"]
    )
    start_epoch, best, stale, history = 0, np.inf, 0, []
    best_state = copy.deepcopy(network.state_dict())
    if path.exists():
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
        if checkpoint["complete"]:
            network.load_state_dict(checkpoint["best_state"])
            return network, checkpoint["report"]
        network.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        start_epoch, best, stale = checkpoint["epoch"] + 1, checkpoint["best"], checkpoint["stale"]
        best_state, history = checkpoint["best_state"], checkpoint["history"]
        torch.set_rng_state(checkpoint["torch_rng"].cpu())
        if DEVICE.type == "cuda":
            torch.cuda.set_rng_state_all([x.cpu() for x in checkpoint["cuda_rng"]])
    x, y, groups = month_batches(train)
    valid_groups = [(c, m, i) for c, m, i in groups if np.isfinite(y[i]).any()]
    counts = pd.Series([c for c, _, _ in valid_groups]).value_counts().to_dict()
    assert counts, "No training labels"
    for epoch in range(start_epoch, MAX_EPOCHS):
        network.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0
        for country, _, index in valid_groups:
            features, labels = (
                torch.as_tensor(x[index], device=DEVICE),
                torch.as_tensor(y[index], device=DEVICE),
            )
            prediction = network(features)
            mask = torch.isfinite(labels)
            loss = (prediction[mask] - labels[mask]).square().mean()
            weight = 1 / (len(counts) * counts[country])
            (loss * weight).backward()
            epoch_loss += loss.detach().item() * weight
        torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
        optimizer.step()
        validation_loss = network_validation(network, validation)
        assert np.isfinite(validation_loss), "Non-finite validation loss"
        history.append(
            {"epoch": epoch + 1, "training_mse": epoch_loss, "validation_mse": validation_loss}
        )
        if validation_loss < best:
            best, stale, best_state = validation_loss, 0, copy.deepcopy(network.state_dict())
        else:
            stale += 1
        report = {
            "parameters": parameters,
            "seed": seed,
            "checkpoint": path.name,
            "validation_monthly_mse": best,
            "trainable_weights": sum(p.numel() for p in network.parameters()),
            "loss_curve": history,
        }
        complete = stale >= PATIENCE or epoch + 1 == MAX_EPOCHS
        state = {
            "complete": complete,
            "epoch": epoch,
            "best": best,
            "stale": stale,
            "model": network.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_state": best_state,
            "history": history,
            "report": report,
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [],
        }
        atomic_checkpoint(state, path)
        if epoch == 0 or (epoch + 1) % 10 == 0 or complete:
            print(f"{tag}: epoch {epoch + 1}, validation monthly MSE {validation_loss:.6g}")
        if complete:
            break
    network.load_state_dict(best_state)
    atomic_json(report, RESULTS / "training" / f"{tag}.json")
    return network, report

## PART 8 — Transfer Learning
### 8.1 One architecture-selection exercise, then matched transfer experiments
Tune up to eight configurations on U.S. training/validation, seed 11. Select once and freeze
the architecture and pretraining settings for every country/history and all approaches.
This keeps matched model structures identical. Local training and pooling use that frozen
configuration with local validation early stopping. Fine-tuning tries three smaller learning
rates (0.03, 0.1, 0.3 times pretraining LR) using local validation; all three seeds are then
run at the chosen rate. The winning first-seed tuning checkpoint is reused. Thus no neural tuning exercise exceeds 20 candidate settings.
Zero-shot weights, architecture and stopping point are selected using U.S. validation only.
No USA observations after target December 2017 enter any gradient update.

In [ ]:
us_train = partition(analysis_countries["USA"], "train")
us_validation = partition(analysis_countries["USA"], "validation")
us_tuning = []
for trial, parameters in enumerate(NN_GRID):
    network, report = fit_network(us_train, us_validation, parameters, SEEDS[0], f"US_tune_{trial}")
    us_tuning.append({"trial": trial, **report})
best_us_trial = min(range(len(us_tuning)), key=lambda i: us_tuning[i]["validation_monthly_mse"])
chosen_parameters = NN_GRID[best_us_trial].copy()
atomic_json(us_tuning, RESULTS / "training/us_tuning.json")
print("Frozen Transformer parameters:", chosen_parameters)
us_states, us_checkpoints, neural_reports = {}, {}, []
for seed in SEEDS:
    tag = f"US_tune_{best_us_trial}" if seed == SEEDS[0] else f"US_pretrain_seed{seed}"
    network, report = fit_network(us_train, us_validation, chosen_parameters, seed, tag)
    us_checkpoints[seed] = tag
    us_states[seed] = {k: v.detach().cpu().clone() for k, v in network.state_dict().items()}
    neural_reports.append({"country": "USA", "history": 18, "approach": "pretrain", **report})
del network

In [ ]:
def choose_adaptation(train, validation, country, years):
    reports = []
    factors = [0.03, 0.1, 0.3] if RUN_MODE == "full" else [0.1]
    for factor in factors:
        params = {**chosen_parameters, "lr": chosen_parameters["lr"] * factor}
        tag = f"{country}_{years}_adapt_tune_{factor}"
        _, report = fit_network(train, validation, params, SEEDS[0], tag, us_states[SEEDS[0]])
        reports.append({**report, "tag": tag})
    best = min(reports, key=lambda report: report["validation_monthly_mse"])
    atomic_json(reports, RESULTS / "training" / f"{country}_{years}_adaptation_tuning.json")
    return best["parameters"], best["tag"]

In [ ]:
for country, years in EXPERIMENTS:
    train, validation, test = local_splits(country, years)
    adaptation_parameters, adaptation_tag = choose_adaptation(train, validation, country, years)
    pooled = pd.concat([us_train, train], ignore_index=True)
    for seed in SEEDS:
        for approach in NEURAL_APPROACHES:
            if approach == "zero_shot":
                network = make_network(chosen_parameters)
                network.load_state_dict(us_states[seed])
                report = {
                    "parameters": chosen_parameters,
                    "seed": seed,
                    "validation_monthly_mse": network_validation(network, validation),
                    "trainable_weights": sum(p.numel() for p in network.parameters()),
                    "checkpoint": f"{us_checkpoints[seed]}.pt",
                }
            else:
                training = pooled if approach == "pooled" else train
                initial = us_states[seed] if approach == "fine_tuned" else None
                params = adaptation_parameters if approach == "fine_tuned" else chosen_parameters
                tag = (
                    adaptation_tag
                    if approach == "fine_tuned" and seed == SEEDS[0]
                    else prediction_tag(country, years, approach, seed)
                )
                network, report = fit_network(training, validation, params, seed, tag, initial)
            for split, sample in [("validation", validation), ("test", test)]:
                save_predictions(
                    network, sample, prediction_tag(country, years, approach, seed), split
                )
            neural_reports.append(
                {"country": country, "history": years, "approach": approach, **report}
            )
            del network
    print(f"Transfer comparisons complete: {country}, {years} year(s)")
    del pooled
atomic_json(baseline_reports + neural_reports, RESULTS / "model_reports.json")

In [ ]:
architecture_table = pd.DataFrame(
    [
        {
            k: r.get(k)
            for k in [
                "country",
                "history",
                "approach",
                "seed",
                "parameters",
                "trainable_weights",
                "validation_monthly_mse",
            ]
        }
        for r in baseline_reports + neural_reports
    ]
)
architecture_table.to_csv(RESULTS / "tables/table_3_models.csv", index=False)
display(architecture_table.head(12))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for report in neural_reports:
    if (
        report["country"] != "GBR"
        or report["history"] != 1
        or report["seed"] != SEEDS[0]
        or "loss_curve" not in report
    ):
        continue
    curve = pd.DataFrame(report["loss_curve"])
    axes[0].plot(curve.epoch, curve.training_mse, label=report["approach"])
    axes[1].plot(curve.epoch, curve.validation_mse, label=report["approach"])
for ax, title in zip(axes, ["Training loss", "Validation loss"]):
    ax.set(title=title + " — UK, 1 year, seed 11", xlabel="Epoch", ylabel="Equal-month MSE")
    ax.legend()
save_figure("09_training_losses")

## PART 9 — Ensemble
### 9.1 Fixed equal average
First average the three neural seeds within each approach; then average **six** constituent
forecasts equally: Elastic Net, XGBoost, local Transformer, zero-shot, fine-tuned and pooled.
This gives each approach one vote. This rule is fixed before inspecting test performance.
Require exact stock-month keys and equal actual returns (including identical missingness).
Seed-level predictions remain saved for variation reporting.

In [ ]:
def equal_average(frames):
    ordered = [f.sort_values(KEY).reset_index(drop=True) for f in frames]
    assert ordered
    reference = ordered[0]
    for frame in ordered[1:]:
        pd.testing.assert_frame_equal(
            reference[KEY + ["target_month", TARGET]], frame[KEY + ["target_month", TARGET]]
        )
    result = reference.copy()
    result["prediction"] = np.mean([f.prediction.to_numpy() for f in ordered], axis=0)
    return result


def collect_forecasts(country, years, split):
    base = RESULTS / "predictions"
    forecasts = {
        name: pd.read_parquet(base / f"{prediction_tag(country, years, name)}_{split}.parquet")
        for name in ["ElasticNet", "XGBoost"]
    }
    for approach in NEURAL_APPROACHES:
        frames = [
            pd.read_parquet(base / f"{prediction_tag(country, years, approach, s)}_{split}.parquet")
            for s in SEEDS
        ]
        forecasts[approach] = equal_average(frames)
    forecasts["ensemble"] = equal_average(list(forecasts.values()))
    return forecasts

In [ ]:
forecasts = {}
for country, years in EXPERIMENTS:
    for split in ["validation", "test"]:
        outputs = collect_forecasts(country, years, split)
        for approach, frame in outputs.items():
            frame.to_parquet(
                RESULTS / "predictions" / f"{country}_{years}_{approach}_average_{split}.parquet",
                index=False,
            )
        if split == "test":
            forecasts[country, years] = outputs
print("Saved equal-average forecasts and constituent predictions.")

## PART 10 — Model Evaluation
### 10.1 Prediction metrics on matching observed outcomes
Pooled MSE weights stock-months equally; monthly MSE first averages within month.
Pooled OOS R² is `1−sum(error²)/sum(actual²)` against zero excess return; month-weighted
R² replaces both sums with equal-month averages. Pearson correlation is pooled; IC is
Spearman correlation within each month, averaged equally. Undefined constant-series ICs
are reported as missing with the number of usable months. IC IR uses √12·mean(IC)/sd(IC).
The test endpoint is verified from nonmissing outcomes, not assumed from calendar dates.

In [ ]:
def monthly_prediction_metrics(frame):
    rows = []
    for month, sample in frame.dropna(subset=[TARGET]).groupby("target_month"):
        y, p = sample[TARGET].to_numpy(), sample.prediction.to_numpy()
        ic = spearmanr(y, p).statistic if len(y) > 1 and np.std(y) > 0 and np.std(p) > 0 else np.nan
        rows.append(
            {
                "target_month": month,
                "mse": np.mean((y - p) ** 2),
                "zero_mse": np.mean(y**2),
                "ic": ic,
                "stocks": len(y),
            }
        )
    return pd.DataFrame(rows).set_index("target_month").sort_index()


def predictive_metrics(frame):
    observed = frame.dropna(subset=[TARGET])
    y, p = observed[TARGET].to_numpy(), observed.prediction.to_numpy()
    monthly = monthly_prediction_metrics(frame)
    denominator = np.sum(y * y)
    return {
        "mse_stock_weighted": np.mean((y - p) ** 2),
        "mse_month_weighted": monthly.mse.mean(),
        "r2_zero_stock_weighted": 1 - np.sum((y - p) ** 2) / denominator if denominator else np.nan,
        "r2_zero_month_weighted": 1 - monthly.mse.mean() / monthly.zero_mse.mean()
        if monthly.zero_mse.mean()
        else np.nan,
        "pearson": np.corrcoef(y, p)[0, 1] if np.std(p) > 0 and np.std(y) > 0 else np.nan,
        "mean_ic": monthly.ic.mean(),
        "ic_std": monthly.ic.std(),
        "ic_ir": np.sqrt(12) * monthly.ic.mean() / monthly.ic.std()
        if monthly.ic.std() > 0
        else np.nan,
        "ic_months": int(monthly.ic.notna().sum()),
        "observed_rows": len(observed),
        "missing_labels": int(frame[TARGET].isna().sum()),
        "first_test_target": observed.target_month.min(),
        "last_test_target": observed.target_month.max(),
    }

In [ ]:
metric_rows, seed_rows = [], []
for (country, years), outputs in forecasts.items():
    equal_average(list(outputs.values()))  # Verify identical comparison samples.
    for approach, frame in outputs.items():
        metric_rows.append(
            {
                "country": country,
                "history": years,
                "approach": approach,
                **predictive_metrics(frame),
            }
        )
        monthly_prediction_metrics(frame).to_csv(
            RESULTS / "tables" / f"monthly_{country}_{years}_{approach}.csv"
        )
    for approach, seed in itertools.product(NEURAL_APPROACHES, SEEDS):
        frame = pd.read_parquet(
            RESULTS
            / "predictions"
            / f"{prediction_tag(country, years, approach, seed)}_test.parquet"
        )
        seed_rows.append(
            {
                "country": country,
                "history": years,
                "approach": approach,
                "seed": seed,
                **predictive_metrics(frame),
            }
        )
metrics = pd.DataFrame(metric_rows)
seed_metrics = pd.DataFrame(seed_rows)
metrics.to_csv(RESULTS / "tables/table_4_predictive_performance.csv", index=False)
seed_metrics.to_csv(RESULTS / "tables/table_4_seed_metrics.csv", index=False)
seed_metrics.groupby(["country", "history", "approach"])[["mse_month_weighted", "mean_ic"]].agg(
    ["mean", "std"]
).to_csv(RESULTS / "tables/table_4_seed_variation.csv")
display(metrics.head(14))

### 10.2 Portfolio formation and missing-outcome policy
Select floor(10%·N) stocks per side from the full ex-ante forecast universe, at least one
stock per side, using stock ID only to break tied predictions deterministically. Each side
has USD notional 1, giving gross exposure 2. Do not subtract the risk-free rate again:
long excess minus short excess already cancels it.

A selected holding with a missing realized return makes that portfolio-month unavailable;
we never drop that stock and re-rank with future knowledge. Report coverage and evaluate
model pairs only on common available months. Cumulative curves stop at missing months;
no missing return is replaced by zero.

Turnover is **total traded notional per dollar of reference capital**, `Σ|w_new−w_old|`.
Use current predictor-month total USD `ret` to drift prior positions if present and known
for every prior holding; drift is not renormalized, representing fixed reference dollar
notional and a cash account. Otherwise use the documented target-weight change proxy for
that rebalance and flag it. Cost is bps/10,000 times this traded notional. Include initial
entry and final liquidation; no borrow fees, market impact, or financing costs are inferred.

In [ ]:
def decile_weights(frame):
    assert len(frame) >= 10, "Fewer than ten forecast stocks in a country-month"
    frame = frame.sort_values(["prediction", "id"], kind="stable")
    count = max(1, len(frame) // 10)
    weights = pd.Series(0.0, index=frame.id.astype(str))
    weights.loc[frame.id.iloc[:count].astype(str)] = -1 / count
    weights.loc[frame.id.iloc[-count:].astype(str)] = 1 / count
    assert np.isclose(weights.sum(), 0) and np.isclose(weights.abs().sum(), 2)
    return weights


def traded_notional(current, previous):
    union = current.index.union(previous.index)
    return float(
        (current.reindex(union, fill_value=0) - previous.reindex(union, fill_value=0)).abs().sum()
    )

In [ ]:
def portfolio_returns(predictions, source_frame):
    previous = pd.Series(dtype=float)
    rows = []
    groups = list(predictions.groupby("target_month", sort=True))
    for position, (target_month, sample) in enumerate(groups):
        weights = decile_weights(sample)
        prior = previous.copy()
        drift_used = not len(prior)
        if len(prior) and "ret" in source_frame:
            predictor_month = target_month - pd.offsets.MonthEnd(1)
            realized = source_frame.loc[source_frame.eom.eq(predictor_month)].set_index("id")["ret"]
            realized.index = realized.index.astype(str)
            drift = pd.to_numeric(realized.reindex(prior.index), errors="coerce")
            if np.isfinite(drift).all() and (drift >= -1).all():
                prior = prior * (1 + drift)
                drift_used = True
        turnover = traded_notional(weights, prior)
        actual = sample.set_index("id")[TARGET]
        actual.index = actual.index.astype(str)
        holdings = weights[weights.ne(0)]
        actual = actual.reindex(holdings.index)
        available = actual.notna().all()
        gross = float((holdings * actual).sum()) if available else np.nan
        liquidation = 0.0
        if position == len(groups) - 1:
            # Use next-month observed total return only for terminal post-return liquidation costs.
            if "ret" in source_frame:
                endpoint = source_frame.loc[source_frame.eom.eq(target_month)].set_index("id")[
                    "ret"
                ]
                endpoint.index = endpoint.index.astype(str)
                end_returns = pd.to_numeric(endpoint.reindex(holdings.index), errors="coerce")
                liquidation = (
                    float((holdings * (1 + end_returns)).abs().sum())
                    if np.isfinite(end_returns).all()
                    else 2.0
                )
            else:
                liquidation = 2.0
            turnover += liquidation
        rows.append(
            {
                "target_month": target_month,
                "gross": gross,
                "traded_notional": turnover,
                "drift_used": drift_used,
                "missing_holdings": int(actual.isna().sum()),
                "terminal_liquidation": liquidation,
                "forecast_stocks": len(sample),
            }
        )
        previous = holdings
    result = pd.DataFrame(rows).set_index("target_month")
    for cost in COST_BPS:
        result[f"net_{cost}bps"] = result.gross - cost / 10_000 * result.traded_notional
    return result

In [ ]:
def annual_sharpe(returns):
    returns = np.asarray(returns, dtype=float)
    return (
        np.sqrt(12) * returns.mean() / returns.std(ddof=1)
        if len(returns) > 1 and returns.std(ddof=1) > 0
        else np.nan
    )

In [ ]:
portfolios, portfolio_rows = {}, []
for (country, years), outputs in forecasts.items():
    for approach, frame in outputs.items():
        result = portfolio_returns(frame, countries[country])
        portfolios[country, years, approach] = result
        result.to_csv(RESULTS / "tables" / f"portfolio_{country}_{years}_{approach}.csv")
    common = (
        pd.concat({a: portfolios[country, years, a].gross for a in outputs}, axis=1)
        .notna()
        .all(axis=1)
    )
    for approach in outputs:
        result = portfolios[country, years, approach]
        sample = result.loc[common]
        for cost in COST_BPS:
            net = sample[f"net_{cost}bps"]
            portfolio_rows.append(
                {
                    "country": country,
                    "history": years,
                    "approach": approach,
                    "cost_bps": cost,
                    "annual_mean_return": 12 * net.mean(),
                    "annual_volatility": np.sqrt(12) * net.std(),
                    "sharpe": annual_sharpe(net),
                    "monthly_traded_notional": sample.traded_notional.mean(),
                    "annual_cost_drag": 12 * cost / 10_000 * sample.traded_notional.mean(),
                    "common_months": len(sample),
                    "unavailable_months": int((~common).sum()),
                    "drift_proxy_months": int((~result.drift_used).sum()),
                }
            )
portfolio_table = pd.DataFrame(portfolio_rows)
portfolio_table.to_csv(RESULTS / "tables/table_6_portfolios.csv", index=False)
display(portfolio_table.head(14))

### 10.3 Predictive and economic comparisons
Annual mean return is 12 times the arithmetic monthly mean; Sharpe assumes monthly
frequency without a serial-correlation adjustment. Block bootstrap below addresses temporal
dependence for paired comparisons. Long-short cumulative wealth assumes profits are
reinvested; insolvency (`1+r≤0`) invalidates this wealth interpretation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for i, country in enumerate(TARGET_COUNTRIES):
    subset = metrics.loc[metrics.country.eq(country)]
    sns.lineplot(data=subset, x="history", y="mean_ic", hue="approach", marker="o", ax=axes[0, i])
    axes[0, i].set(title=f"{country}: test mean monthly IC", xlabel="Local training years")
    subset = portfolio_table.loc[
        portfolio_table.country.eq(country) & portfolio_table.cost_bps.eq(10)
    ]
    sns.lineplot(data=subset, x="history", y="sharpe", hue="approach", marker="o", ax=axes[1, i])
    axes[1, i].set(title=f"{country}: test Sharpe at 10 bps", xlabel="Local training years")
save_figure("10_history_comparisons")
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for i, country in enumerate(TARGET_COUNTRIES):
    for approach in forecasts[country, 1]:
        returns = portfolios[country, 1, approach]
        for j, series in enumerate([returns.gross, returns.net_10bps]):
            cumulative = (1 + series).cumprod(skipna=False) - 1
            axes[i, j].plot(cumulative.index, cumulative, label=approach)
            axes[i, j].set(
                title=f"{country}, 1-year history: {'gross' if j == 0 else '10 bps net'}",
                ylabel="Cumulative long-short return",
                xlabel="Test target month",
            )
    axes[i, 0].legend(fontsize=8)
save_figure("11_cumulative_portfolios")

## PART 11 — Statistical Analysis
### 11.1 Six-month moving-block bootstrap, paired on common months
Compare every model pair including the ensemble for monthly MSE, monthly IC, gross return,
10 bps net return and 10 bps net Sharpe. Resample the **same** contiguous six-month blocks
for both models; blocks never bridge a calendar gap. Percentile CIs describe paired
performance differences; two-sided bootstrap p-values use the centered bootstrap null
(`difference* − observed_difference`). Holm correction covers **all pairs, countries,
histories and metrics together**. This is conservative and declared before testing.
Fewer than 12 common months or absent six-month blocks gives an explicit unavailable test.
Full mode uses 49,999 resamples so its p-value resolution can support the 630-test Holm family.
Smoke p-values are diagnostics only. MSE differences below zero favor model A; positive
IC/return/Sharpe differences favor A. No seed is treated as an independent market sample.

In [ ]:
def moving_block_indices(dates, repetitions, rng, block_length=6):
    dates = pd.DatetimeIndex(dates)
    ordinals = dates.to_period("M").astype(int).to_numpy()
    starts = [
        i
        for i in range(len(dates) - block_length + 1)
        if np.all(np.diff(ordinals[i : i + block_length]) == 1)
    ]
    if len(dates) < 12 or not starts:
        raise ValueError("Insufficient contiguous common evaluation months")
    choices = rng.choice(starts, size=(repetitions, int(np.ceil(len(dates) / block_length))))
    return (choices[:, :, None] + np.arange(block_length)).reshape(repetitions, -1)[:, : len(dates)]


def paired_bootstrap(a, b, statistic="mean", repetitions=None, seed=711):
    joined = pd.concat([a.rename("a"), b.rename("b")], axis=1, sort=True).dropna().sort_index()
    repetitions = BOOTSTRAPS if repetitions is None else repetitions
    try:
        indices = moving_block_indices(joined.index, repetitions, np.random.default_rng(seed))
    except ValueError as error:
        return {
            "difference": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "p_value": np.nan,
            "months": len(joined),
            "status": str(error),
        }
    x, y = joined.a.to_numpy(), joined.b.to_numpy()
    if statistic == "sharpe":
        difference = annual_sharpe(x) - annual_sharpe(y)
        xb, yb = x[indices], y[indices]
        with np.errstate(divide="ignore", invalid="ignore"):
            samples = np.sqrt(12) * (
                xb.mean(axis=1) / xb.std(axis=1, ddof=1) - yb.mean(axis=1) / yb.std(axis=1, ddof=1)
            )
    else:
        difference = float(np.mean(x - y))
        samples = (x[indices] - y[indices]).mean(axis=1)
    samples = samples[np.isfinite(samples)]
    if not np.isfinite(difference) or len(samples) < repetitions * 0.9:
        return {
            "difference": difference,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "p_value": np.nan,
            "months": len(joined),
            "status": "Undefined bootstrap statistic",
        }
    low, high = np.quantile(samples, [0.025, 0.975])
    p = (1 + np.sum(np.abs(samples - difference) >= abs(difference))) / (len(samples) + 1)
    return {
        "difference": difference,
        "ci_low": low,
        "ci_high": high,
        "p_value": p,
        "months": len(joined),
        "status": "ok",
    }

In [ ]:
comparison_rows = []
for (country, years), outputs in forecasts.items():
    monthly = {a: monthly_prediction_metrics(f) for a, f in outputs.items()}
    shared_portfolio_months = (
        pd.concat({a: portfolios[country, years, a].gross for a in outputs}, axis=1).dropna().index
    )
    for a, b in itertools.combinations(outputs, 2):
        for metric in ["mse", "ic", "gross", "net_10bps", "net_sharpe_10bps"]:
            if metric in ["mse", "ic"]:
                left, right = monthly[a][metric], monthly[b][metric]
            else:
                column = "net_10bps" if metric == "net_sharpe_10bps" else metric
                left = portfolios[country, years, a].loc[shared_portfolio_months, column]
                right = portfolios[country, years, b].loc[shared_portfolio_months, column]
            comparison_rows.append(
                {
                    "country": country,
                    "history": years,
                    "model_a": a,
                    "model_b": b,
                    "metric": metric,
                    **paired_bootstrap(
                        left, right, "sharpe" if metric == "net_sharpe_10bps" else "mean"
                    ),
                }
            )
statistics = pd.DataFrame(comparison_rows)
valid = statistics.p_value.notna()
statistics["p_holm"], statistics["significant_5pct"] = np.nan, False
if valid.any():
    rejected, corrected, _, _ = multipletests(statistics.loc[valid, "p_value"], method="holm")
    statistics.loc[valid, "p_holm"] = corrected
    statistics.loc[valid, "significant_5pct"] = rejected
statistics.to_csv(RESULTS / "tables/table_7_significance.csv", index=False)
display(statistics.head(15))

## PART 12 — Research Conclusions
### 12.1 Transfer comparisons and conditional findings
The results below report computed evidence, with effect directions and multiplicity-adjusted
p-values. It does not declare success from the largest point estimate. A nonsignificant
result is inconclusive; it does not prove equality. Increasing-history effects are
reported descriptively and do not inherit the significance of within-history tests.

In [ ]:
local_reference = metrics.loc[
    metrics.approach.eq("local"), ["country", "history", "mse_month_weighted", "mean_ic"]
]
transfer = metrics.merge(
    local_reference, on=["country", "history"], suffixes=("", "_local"), validate="many_to_one"
)
transfer["mse_change_vs_local"] = transfer.mse_month_weighted - transfer.mse_month_weighted_local
transfer["ic_change_vs_local"] = transfer.mean_ic - transfer.mean_ic_local
transfer.to_csv(RESULTS / "tables/table_5_transfer_vs_local.csv", index=False)
display(transfer[["country", "history", "approach", "mse_change_vs_local", "ic_change_vs_local"]])

In [ ]:
def comparison_evidence(country, years, a, b, metric="mse"):
    row = statistics.loc[
        statistics.country.eq(country)
        & statistics.history.eq(years)
        & statistics.metric.eq(metric)
        & (
            (statistics.model_a.eq(a) & statistics.model_b.eq(b))
            | (statistics.model_a.eq(b) & statistics.model_b.eq(a))
        )
    ].iloc[0]
    sign = 1 if row.model_a == a else -1
    difference = row.difference * sign
    favored = a if (difference < 0 if metric == "mse" else difference > 0) else b
    if not np.isfinite(difference):
        finding = "Insufficient common observations for inference"
    elif row.significant_5pct:
        finding = f"Evidence favors {favored} after Holm correction"
    else:
        finding = f"Numerically favors {favored}; inconclusive after Holm correction"
    return {
        "country": country,
        "history": years,
        "comparison": f"{a} minus {b}",
        "metric": metric,
        "difference": difference,
        "p_holm": row.p_holm,
        "finding": finding,
    }

In [ ]:
finding_rows = []
for country, years in EXPERIMENTS:
    for a, b in [
        ("fine_tuned", "local"),
        ("zero_shot", "local"),
        ("fine_tuned", "zero_shot"),
        ("fine_tuned", "pooled"),
        ("local", "ElasticNet"),
        ("local", "XGBoost"),
        ("ensemble", "fine_tuned"),
    ]:
        finding_rows.append(comparison_evidence(country, years, a, b))
    finding_rows.append(comparison_evidence(country, years, "fine_tuned", "local", "net_10bps"))
findings = pd.DataFrame(finding_rows)
if RUN_MODE == "smoke":
    findings["finding"] = "SMOKE DIAGNOSTIC ONLY — no research conclusion"
findings.to_csv(RESULTS / "tables/table_8_research_findings.csv", index=False)
display(findings)
for country in TARGET_COUNTRIES:
    changes = transfer.loc[transfer.country.eq(country) & transfer.approach.eq("fine_tuned")]
    print(
        country,
        "fine-tuning minus local MSE by history:",
        changes.set_index("history").mse_change_vs_local.to_dict(),
    )
    print(
        "Whether this pattern strengthens or weakens with history is descriptive, not a tested interaction."
    )

In [ ]:
report_lines = [
    f"# {'SMOKE DIAGNOSTICS — NOT RESEARCH RESULTS' if RUN_MODE == 'smoke' else 'Research evidence'}",
    f"Run: {RUN_ID}; frozen predictor endpoint: {manifest['end']}",
    "All model comparisons use identical forecast keys and matching observed labels.",
]
for _, row in findings.iterrows():
    report_lines.append(
        f"- {row.country}, {row.history} year(s), {row.comparison}, {row.metric}: "
        f"difference={row.difference:.6g}, Holm p={row.p_holm:.4g}. {row.finding}."
    )
for country in TARGET_COUNTRIES:
    for approach in ["fine_tuned", "ensemble"]:
        sample = portfolio_table.loc[
            portfolio_table.country.eq(country)
            & portfolio_table.approach.eq(approach)
            & portfolio_table.cost_bps.eq(10)
        ]
        for _, row in sample.iterrows():
            report_lines.append(
                f"- {country}, {row.history} years, {approach}: 10 bps net annual arithmetic mean "
                f"{row.annual_mean_return:.2%}, Sharpe {row.sharpe:.3f}, {row.common_months} common months. "
                "These are point estimates, not proof of profitability."
            )
(RESULTS / "research_findings.md").write_text("\n\n".join(report_lines))
print("All outputs saved to:", RESULTS)

### 12.2 Interpretation and limitations
Read Tables 4–8 together. For UK and Japan, compare fine-tuned with local for pretraining
value; zero-shot with local for direct transfer; fine-tuned with zero-shot for adaptation;
and fine-tuned with pooled for transfer beyond joint training. Compare local attention to
both baselines, and the ensemble to each constituent using Table 7. Table 5 reports whether
transfer advantages change with history. Table 6 answers whether observed predictive gains
coincide with positive net portfolio means, without asserting future profitability.

* The dataset release and available endpoint can differ from proposal counts. WRDS access
  and historical revisions constrain reproducibility across independently acquired snapshots.
* Shared feature selection uses the shortest allowed local windows, holding inputs fixed.
  Categorical vocabularies use permitted training data; rankings use current cross-sections.
* Validation selection is repeated across experiments. The held-out test and global Holm
  adjustment limit overclaiming but do not remove all researcher degrees of freedom.
* Three seeds measure optimization variation, not independent economic samples. The shared
  U.S.-selected architecture prioritizes fair transfer comparisons over per-market tuning.
* Missing return labels can make portfolios unavailable; coverage and drift-proxy counts are
  reported. Missing outcomes are never silently replaced with zero or used to select holdings.
* USD returns combine local equity and FX exposure. The target already subtracts U.S. bills.
* Decile backtests omit realistic borrow/financing constraints, short availability, market
  impact, taxes and capacity. Cost sensitivities are assumptions, not measured execution costs.
* Full attention has quadratic compute requirements. Only a full monthly benchmark on the
  allocated hardware establishes feasibility. Smoke mode is not a scalability result.
* Six-month block length is fixed by the proposal; bootstrap inference is approximate,
  particularly with short or discontinuous samples. Holm controls a broad declared family.

### 12.3 Execution and reproducibility
See the short `README.md` for setup and `amarel_job.sbatch` for cluster execution.
Do not run full training on an Amarel login node. Full-data results remain pending until
this notebook is executed with authenticated real WRDS data and sufficient computing time.